In [0]:
%sql

-- Create catalog for the entire project
CREATE CATALOG IF NOT EXISTS ecommerce_pipeline
COMMENT 'Analytics Engineering Portfolio Project';

-- Create schemas for each pipeline layer  
CREATE SCHEMA IF NOT EXISTS ecommerce_pipeline.raw
COMMENT 'Raw data as ingested from source';

CREATE SCHEMA IF NOT EXISTS ecommerce_pipeline.staging
COMMENT 'Cleaned and typed data';

CREATE SCHEMA IF NOT EXISTS ecommerce_pipeline.intermediate
COMMENT 'Business logic applied';

CREATE SCHEMA IF NOT EXISTS ecommerce_pipeline.mart
COMMENT 'Final tables for reporting';

In [0]:
%sql
SHOW SCHEMAS IN ecommerce_pipeline;

databaseName
default
information_schema
intermediate
mart
raw
staging


In [0]:
%sql

-- ============================================================
-- Load all raw events into single partitioned Delta table
-- Source: all CSV files in Volume (landing zone)
-- Target: ecommerce_pipeline.raw.events
-- Partitioned by year and month for query performance
-- ============================================================

CREATE TABLE IF NOT EXISTS ecommerce_pipeline.raw.events
COMMENT 'Raw eCommerce behavioral events. Source: Kaggle mkechinov dataset.'
PARTITIONED BY (year, month)
AS
SELECT 
    *,
    YEAR(event_time)  AS year,
    MONTH(event_time) AS month
FROM read_files(
    '/Volumes/workspace/default/ecommerce_raw/',
    format => 'csv',
    header => true,
    inferSchema => true
);

num_affected_rows,num_inserted_rows


In [0]:
%sql
SELECT COUNT(*) FROM ecommerce_pipeline.raw.events;

COUNT(*)
109950743


In [0]:
%sql

-- Check table schema and data types
DESCRIBE TABLE ecommerce_pipeline.raw.events;

col_name,data_type,comment
event_time,timestamp,null
event_type,string,null
product_id,int,null
category_id,bigint,null
category_code,string,null
brand,string,null
price,double,null
user_id,int,null
user_session,string,null
_rescued_data,string,null


In [0]:
%sql

-- Check if any data was rescued (failed to parse)
SELECT COUNT(*) as rescued_count
FROM ecommerce_pipeline.raw.events
WHERE _rescued_data IS NOT NULL;

rescued_count
0


In [0]:
%sql

-- Preview first 5 rows
SELECT * EXCEPT(_rescued_data)
FROM ecommerce_pipeline.raw.events
LIMIT 5;

event_time,event_type,product_id,category_id,category_code,brand,price,user_id,user_session,year,month
2019-10-20T12:51:37.000Z,view,15200217,2053013553484398879,null,null,123.56,545081263,c2e016df-d267-4cf6-bf66-eda903df1610,2019,10
2019-10-20T12:51:38.000Z,view,1004835,2053013555631882655,electronics.smartphone,samsung,224.65,560507953,4829f365-e377-4d78-b64a-7d5b1303ab5b,2019,10
2019-10-20T12:51:38.000Z,view,18001180,2053013558525952589,null,samsung,1.45,560213179,8b3dc502-3774-45d7-a9c1-d198895d791c,2019,10
2019-10-20T12:51:38.000Z,view,12714795,2053013553559896355,null,null,57.0,534104430,de7fdd38-d403-49d5-abeb-b762c617230e,2019,10
2019-10-20T12:51:38.000Z,view,12200523,2116907525176557699,sport.bicycle,phoenix,128.68,527897602,69a786e8-0de3-4e4e-b178-d1876d9afb96,2019,10


In [0]:
%sql

-- Show all tables in raw schema
SHOW TABLES IN ecommerce_pipeline.raw;

database,tableName,isTemporary
raw,events,false
